# Text baseline

Use the full main dataset. Make one document per user, then predict the five
Big Five scores with TF-IDF and Ridge. Run the cells in order.

In [1]:
from pathlib import Path
from tempfile import TemporaryDirectory

import pandas as pd
from scipy.stats import pearsonr
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import GridSearchCV
from sklearn.pipeline import make_pipeline

DATA = Path("../data/preprocessed")  # From the project's src folder
OUTPUT = Path("../data/text_baseline")
TRAITS = ["agreeableness", "openness", "conscientiousness",
          "extraversion", "neuroticism"]
ALPHAS = [0.01, 0.1, 1, 10, 100]

## 1. Make one document per user

Joining comments does not learn anything from other users. The model learns
the vocabulary and IDF later, using only its training users.

In [2]:
users = pd.read_csv(DATA / "main/users.csv").set_index("author")
outer = pd.read_csv(DATA / "outer_folds.csv").set_index("author")
inner = pd.read_csv(DATA / "inner_folds.csv")
assert users.index.is_unique and outer.index.is_unique
assert set(users.index) == set(outer.index)

comments = pd.read_csv(DATA / "main/comments.csv", usecols=["author", "body"],
                       keep_default_na=False)
documents = comments.groupby("author")["body"].agg(" ".join)
del comments

data = users[TRAITS].join(outer).join(documents.rename("text"))
assert data["text"].notna().all()
print("Users:", len(data))

Users: 1311


## 2. Train and predict

For each outer fold, pick alpha separately for each trait by inner validation
RMSE. The pipeline fits TF-IDF again each time it trains. We also predict the
outer training mean as a reference.

In [3]:
predictions = []

for fold in sorted(data["outer_fold"].unique()):
    train = data[data["outer_fold"] != fold]
    test = data[data["outer_fold"] == fold]

    inner_fold = inner[inner["outer_fold"] == fold].set_index("author")["inner_fold"]
    assert inner_fold.index.is_unique and set(inner_fold.index) == set(train.index)
    inner_fold = inner_fold.reindex(train.index).to_numpy()
    cv = []
    for val_fold in sorted(set(inner_fold)):
        train_idx = (inner_fold != val_fold).nonzero()[0]
        val_idx = (inner_fold == val_fold).nonzero()[0]
        cv.append((train_idx, val_idx))

    # Ridge can choose a different alpha for each trait. The TF-IDF input
    # is the same for all traits, so cache it for the searches in this fold.
    with TemporaryDirectory() as cache:
        model = make_pipeline(
            TfidfVectorizer(min_df=2, max_features=50_000),
            Ridge(solver="lsqr"),
            memory=cache,
        )

        for trait in TRAITS:
            search = GridSearchCV(model, {"ridge__alpha": ALPHAS}, cv=cv,
                                  scoring="neg_root_mean_squared_error")
            search.fit(train["text"], train[trait])
            predictions.append(pd.DataFrame({
                "author": test.index,
                "outer_fold": fold,
                "trait": trait,
                "true": test[trait].to_numpy(),
                "text_prediction": search.predict(test["text"]),
                "mean_prediction": train[trait].mean(),
                "alpha": search.best_params_["ridge__alpha"],
            }))
    print("Finished fold", fold)

Finished fold 0


Finished fold 1


Finished fold 2


Finished fold 3


Finished fold 4


## 3. Check the results

Each user has one test prediction per trait. Compute the metrics using those
predictions. Pearson correlation is left blank if the predictions are constant.

In [4]:
results = pd.concat(predictions, ignore_index=True)
assert len(results) == len(users) * len(TRAITS)
assert not results.duplicated(["author", "trait"]).any()

metrics = []
for trait, rows in results.groupby("trait"):
    actual = rows["true"]
    for model in ["text", "mean"]:
        predicted = rows[f"{model}_prediction"]
        correlation = float("nan")
        if predicted.nunique() > 1 and actual.nunique() > 1:
            correlation = pearsonr(actual, predicted)[0]
        metrics.append({
            "trait": trait,
            "model": model,
            "rmse": mean_squared_error(actual, predicted) ** 0.5,
            "pearson_r": correlation,
            "r2": r2_score(actual, predicted),
        })

metrics = pd.DataFrame(metrics)
OUTPUT.mkdir(parents=True, exist_ok=True)
results.to_csv(OUTPUT / "predictions.csv", index=False)
metrics.to_csv(OUTPUT / "metrics.csv", index=False)
display(metrics.round(3))

,trait,model,rmse,pearson_r,r2
0,agreeableness,text,29.810,0.243,0.056
1,agreeableness,mean,30.713,-0.054,-0.002
2,conscientiousness,text,29.648,0.177,0.031
3,conscientiousness,mean,30.141,-0.050,-0.001
4,extraversion,text,29.077,0.261,0.066
5,extraversion,mean,30.129,-0.068,-0.003
6,neuroticism,text,31.639,0.195,0.030
7,neuroticism,mean,32.164,-0.061,-0.002
8,openness,text,27.001,0.233,0.054
9,openness,mean,27.778,-0.049,-0.001
